# C.2 最後答對代表過程正確嗎？


回答2+2寫「2+2=5；5−1=4」，結尾4對，第一步卻錯。檢查不能只讀最後一個數字，還要分開查每條等式、前後是否接續，以及是不是在解原題。

每步 `(a,b,claimed)` 是宣稱a+b=claimed，減1寫成加-1。先把這份錯解編成兩個三元組：


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
steps = [(2, 2, 5), (5, -1, 4)]
truth = 2 + 2
valid = [a + b == claimed for a, b, claimed in steps]
linked = [steps[i][0] == steps[i - 1][2] for i in range(1, len(steps))]
print("等式算對", valid)
print("接上前步", linked)
print("最後答案對", steps[-1][2] == truth)

等式算對 [False, True]
接上前步 [True]
最後答案對 True


等式是False、True；接續True；最後答案True。第二步真的從前步聲稱的5開始，卻不能救第一步。只把第一步claimed改4，等式都True，第二步仍從5開始，接續便False。

即使等式與接續都成立，還要核對原題參數與任務（短程式尚未檢查這一項）：無理由地多加一個數再減回，不能只靠算術局部正確就接受。原題其實只需2+2=4。驗證器是依明確規則核對輸出的程式，應說明它驗了哪些條件，不稱通用文字證明器。

舊模型 `(5+0)+3` 生成 `5+0=5;5+3=10;answer=10`，格式和接續都通過，第二條加法仍錯。這個例子讓我們把「能解析」與「算對」分開；通過窄驗證器也不證明輸出是內部推理的忠實記錄。

舊候選的完整紀錄見[正式報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/reasoning.json)。
